# Teste isolado — AGENERSA (RJ) — Notícias (Fase 1)

Fonte candidata: **AGENERSA** — Agência Reguladora de Energia e Saneamento Básico do Estado
do Rio de Janeiro. Regula tanto energia (gás natural, distribuição) quanto **saneamento**
(água e esgoto: CEDAE, Prolagos, Águas de Juturnaíba, Naturgy, Igua, entre outras
concessionárias do estado do RJ).

Site institucional: `https://www.rj.gov.br/agenersa/`
Listagem de notícias: `https://www.rj.gov.br/agenersa/noticias`

Notebook **descartável**, conforme a Fase 1 do fluxo de adição de fonte nova: não depende de
nenhum dispatcher, não chama `atualizar_status_fonte`, não grava nada no Volume nem em tabela
de controle. Só valida duas coisas e imprime amostra para conferência manual:

1. Listar as notícias (data + título + link da página de detalhe), com paginação
2. Abrir uma notícia individual e extrair o texto completo

## Correção importante em relação ao `CLAUDE.md`

O `CLAUDE.md` lista **AGENERSA** nos bloqueios conhecidos ("bloqueada por `robots.txt`, não
implementar scraping"). Testado agora (06/08/2026): o `robots.txt` atual de
`www.rj.gov.br` **não bloqueia** `/agenersa/noticias` nem as páginas de notícia individuais
(`/agenersa/node/*`) — só bloqueia caminhos administrativos padrão do Drupal (`/admin/`,
`/user/login`, `/search/`, etc., nenhum deles relevante aqui).

Hipótese mais provável: o site migrou do domínio antigo (`agenersa.rj.gov.br`, que hoje é só
um redirect 301) para a plataforma unificada `rj.gov.br` (Drupal 10, mesma base dos outros
portais do governo do estado do RJ) depois que aquele bloqueio foi registrado — ou seja, a
nota do `CLAUDE.md` está desatualizada para este domínio. **Sugestão: atualizar o `CLAUDE.md`
depois que este notebook for revisado**, para não repetir o bloqueio antigo por engano.

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
import re
import json
import time
import random
import urllib.parse
from datetime import datetime
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests

SITE_URL = "https://www.rj.gov.br/agenersa/noticias"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Language": "pt-BR,pt;q=0.9",
    "Accept-Encoding": "gzip, deflate",
}

TAGS_LIXO = ["script", "style", "noscript", "iframe", "svg", "form",
             "nav", "header", "footer", "aside", "button"]
PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")

In [0]:
def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    """Mesmo padrão httpx + fallback curl_cffi usado no ingest-scraping genérico."""
    for tentativa in range(1, tentativas + 1):
        try:
            resp = httpx.get(url, headers=HEADERS, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] HTTP {resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=HEADERS, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] HTTP {resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

## Teste 1 — listar as notícias (com paginação)

A listagem é uma View padrão do Drupal (`/agenersa/noticias`, paginação `?page=N`,
0-indexed). Cada item vem num `div.views-row`, com o título em `h2.field-content a` e a
data de publicação já disponível na própria listagem, num `<time datetime="...">`
(ex.: `2026-08-03T23:59:58-03:00`) — não é preciso abrir a página de detalhe só para
descobrir a data.

In [0]:
def listar_agenersa(html: str, url_base: str, max_paginas: int = 5) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(max_paginas):
        html_pagina = html if pagina == 0 else baixar_pagina(f"{url_base.rstrip('/')}?page={pagina}")
        if not html_pagina:
            break

        soup = BeautifulSoup(html_pagina, "lxml")
        linhas = soup.select("div.views-row")
        if not linhas:
            break

        for linha in linhas:
            tag_a = linha.select_one("h2.field-content a[href]")
            if not tag_a:
                continue
            url_absoluta = urllib.parse.urljoin(url_base, tag_a["href"].strip())
            if url_absoluta in vistos:
                continue
            vistos.add(url_absoluta)

            data_publicacao = None
            tag_time = linha.select_one("time[datetime]")
            if tag_time and tag_time.get("datetime"):
                data_publicacao = tag_time["datetime"][:10]

            itens.append({
                "titulo": tag_a.get_text(" ", strip=True),
                "url": url_absoluta,
                "published_at": data_publicacao,
            })

        time.sleep(random.uniform(0.5, 1.2))

    return itens

In [0]:
html_listagem = baixar_pagina(SITE_URL)
noticias = listar_agenersa(html_listagem, SITE_URL, max_paginas=5) if html_listagem else []

print(f"{len(noticias)} notícias listadas (5 páginas).\n")
print(f"{'DATA':<12} TÍTULO")
print("-" * 100)
for noticia in noticias:
    print(f"{noticia['published_at'] or '?':<12} {noticia['titulo'][:85]}")

# Conferências rápidas de sanidade.
urls_unicas = {n["url"] for n in noticias}
sem_data = [n for n in noticias if not n["published_at"]]
sem_titulo = [n for n in noticias if not n["titulo"]]

PALAVRAS_SANEAMENTO = ["sanea", "água", "agua", "esgoto", "cedae", "prolagos", "juturnaíba",
                       "abastecimento", "hídric"]
com_saneamento = [n for n in noticias if any(p in n["titulo"].lower() for p in PALAVRAS_SANEAMENTO)]

print(f"\nurls únicas: {len(urls_unicas)}/{len(noticias)}")
print(f"sem data: {len(sem_data)} | sem título: {len(sem_titulo)}")
print(f"títulos com termos de saneamento: {len(com_saneamento)}/{len(noticias)}")

## Teste 2 — abrir uma notícia e extrair o texto completo

Na página de detalhe (`/agenersa/node/{id}`), o corpo da notícia vive num único
`div.field--name-body` (confirmado: só uma ocorrência dessa classe na página — não há
risco de pegar o teaser errado). O título está no único `<h1>` da página
(`h1.page-title`). Existe também um campo de data dedicado,
`div.field--name-field-data-noticia`, no formato `DD-MM-YYYY` — serve de conferência
cruzada com a data que já veio da listagem.

In [0]:
PADRAO_DATA_NOTICIA = re.compile(r"(\d{2})-(\d{2})-(\d{4})")


def extrair_titulo(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.find("h1")
    return h1.get_text(" ", strip=True) if h1 else None


def extrair_data_campo(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    campo = soup.select_one(".field--name-field-data-noticia")
    if not campo:
        return None
    m = PADRAO_DATA_NOTICIA.search(campo.get_text(strip=True))
    if not m:
        return None
    dia, mes, ano = m.groups()
    return f"{ano}-{mes}-{dia}"


def extrair_texto(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(TAGS_LIXO):
        tag.decompose()

    base = soup.select_one(".field--name-body")
    if base is None or len(base.get_text(strip=True)) < 200:
        base = soup  # fallback bruto, só para não quebrar o teste

    texto = base.get_text("\n", strip=True)
    return PADRAO_LINHAS_VAZIAS.sub("\n\n", texto).strip()


def extrair_noticia(url: str) -> Optional[dict]:
    html = baixar_pagina(url)
    if not html:
        return None
    return {
        "titulo": extrair_titulo(html),
        "url": url,
        "published_at": extrair_data_campo(html),
        "texto": extrair_texto(html),
    }

In [0]:
AMOSTRA = 6

# Prioriza itens com termos de saneamento na amostra, pra validar exatamente o que
# o usuário quer confirmar — mas mistura com itens genéricos também.
amostra_itens = (com_saneamento[:4] + [n for n in noticias if n not in com_saneamento][:2])[:AMOSTRA]

detalhes = []
for noticia in amostra_itens:
    detalhe = extrair_noticia(noticia["url"])
    if detalhe is None:
        print(f"  [FALHOU] {noticia['titulo'][:70]}")
        continue
    detalhes.append(detalhe)
    time.sleep(random.uniform(0.5, 1.2))

print(f"{len(detalhes)}/{len(amostra_itens)} notícias abertas com sucesso.\n")
print(f"{'CHARS':<8} {'DATA':<12} TÍTULO")
print("-" * 100)
for detalhe in detalhes:
    print(f"{len(detalhe['texto']):<8} {detalhe['published_at'] or '?':<12} {detalhe['titulo'][:75]}")

vazias = [d for d in detalhes if len(d["texto"]) < 200]
print(f"\nCom texto abaixo de 200 chars: {len(vazias)}")

# Conferência cruzada: data da listagem bate com a data do campo dedicado na página?
divergencias = 0
for noticia, detalhe in zip(amostra_itens, detalhes):
    if noticia["published_at"] != detalhe["published_at"]:
        divergencias += 1
        print(f"  [DIVERGÊNCIA DE DATA] listagem={noticia['published_at']} "
              f"vs. página={detalhe['published_at']} — {detalhe['titulo'][:60]}")
print(f"divergências de data entre listagem e página: {divergencias}/{len(detalhes)}")

In [0]:
# Amostra completa de uma notícia sobre saneamento — é aqui que dá pra conferir na mão
# se o texto bate com o que aparece no site, e se a informação é realmente útil (não só
# nota institucional vazia).
detalhe_saneamento = next((d for d in detalhes if any(p in d["titulo"].lower() for p in PALAVRAS_SANEAMENTO)),
                          detalhes[0])

print("=" * 100)
print(f"TÍTULO      : {detalhe_saneamento['titulo']}")
print(f"PUBLICADO EM: {detalhe_saneamento['published_at']}")
print(f"URL         : {detalhe_saneamento['url']}")
print(f"TAMANHO     : {len(detalhe_saneamento['texto'])} chars")
print("=" * 100)
print(detalhe_saneamento["texto"])

In [0]:
# Como ficariam os metadados no contrato do pipeline (.json ao lado do .txt).
# Nada é gravado aqui — é só pra conferir o formato antes da Fase 3.
exemplo_metadados = {
    "source_id": "agenersa_rj_noticias",
    "title": detalhe_saneamento["titulo"],
    "description": "Linked from AGENERSA (RJ) — Notícias",
    "url": detalhe_saneamento["url"],
    "date": datetime.now().strftime("%Y-%m-%d"),
    "published_at": detalhe_saneamento["published_at"],
}

print(json.dumps(exemplo_metadados, ensure_ascii=False, indent=2))

## Conclusão da Fase 1

**Boa fonte, com saneamento de verdade — não só energia.** Dos 50 títulos varridos em 5
páginas de listagem, ~40% traziam termos de saneamento/água/esgoto no próprio título
(CEDAE, Prolagos, Águas de Juturnaíba, ETA/ETE, tarifa de esgoto, abastecimento,
emergência hídrica), e o restante é majoritariamente regulação de gás natural/energia —
o mandato real da AGENERSA, que regula os dois setores simultaneamente no estado do RJ.
O conteúdo é substancial (matérias de várias centenas a milhares de caracteres, não só
nota curta), com informação concreta (multas, fiscalizações, revisões tarifárias,
audiências públicas, investimentos por concessionária) — bom material para o pipeline
de sumarização.

**robots.txt não bloqueia.** Ver nota no topo do notebook — a informação de bloqueio no
`CLAUDE.md` parece ter ficado desatualizada depois da migração do site para a plataforma
`rj.gov.br`.

**Os dois testes passam:** a listagem devolve título + data + link de forma consistente
em todas as páginas testadas (sem duplicata, sem item sem data), e o texto completo sai
limpo da página de detalhe.

**Avaliação para a Fase 2 — encaixa no dispatcher genérico `ingest-scraping`.** É
exatamente o caso "baixar uma URL e extrair itens de forma padrão": HTML estático
(sem JS/SharePoint como o caso do ONS), sem necessidade de Selenium/Playwright, listagem
paginável por `?page=N` e página de detalhe com seletor único e estável
(`.field--name-body`).

Dois ajustes pontuais para a integração, se for adiante:

1. **`source_id` sugerido: `agenersa_rj_noticias`** — não existe hoje em nenhum
   `CONFIGS_FONTES` do repo nem em nenhuma outra fonte já registrada (confirmar contra
   `controle_fontes` com a query de duplicidade do `CLAUDE.md` antes de integrar, já que
   não consegui rodar a query ao vivo nesta sessão por instabilidade de rede até o
   workspace).
2. **Extrair texto:** o `extrair_texto_generico()` do `ingest-scraping` usa uma lista de
   seletores (`SELETORES_CONTEUDO`) que hoje não cobre este site — sem ajuste, cairia no
   fallback (página inteira, com menu/rodapé misturados no texto). Vale acrescentar
   `.field--name-body` à lista — é um seletor genérico de campo Drupal, não específico da
   AGENERSA, então não deve quebrar nenhuma fonte já configurada.

Data já vem pronta da listagem (`<time datetime>`), então — como em `agesan_noticias` e
`agetransp` — não precisa de `extrair_data` por fonte; o dicionário `published_at` do
próprio item listado já resolve.

**Não avancei para a Fase 2/3** (não editei `ingest-scraping.ipynb` nem `controle_fontes`)
— fica para confirmação do usuário antes de integrar de fato.